# C9 · Spatial autocorrelation: are neighbours alike?

> **Time:** about 2 hours  
> **You need:** Parts A and B, and Lessons C1 to C8, especially slicing and masks (A8), functions (A5), `touches` (B7), `sjoin` (B9), hexagon grids (B10), the 90 m UTM grid (C4), slope from `np.gradient` (C6), `zonal_stats` (C7) and the first law of geography (C8).  
> **By the end you can:**
> 1. explain positive, zero and negative **spatial autocorrelation**, and measure it on a raster by correlating the grid with a shifted copy of itself (a **correlogram**),
> 2. build **neighbours** and a row-standardised **weights matrix** for polygons, and compute a **spatial lag** and **Moran's I** with your own function, tested with a **permutation test**,
> 3. map clusters of high and low values, and explain why a **random** train/test split of autocorrelated data gives over-optimistic scores, and what a **spatial** split changes.

C8 rested on one idea: near things are more alike than distant things. This lesson **measures** that idea. *How alike are neighbours? Over what distance does the likeness last? Could the pattern be chance?* The answers matter far beyond maps. In D4, and in every machine-learning study of landslides or land cover, they decide whether a test of a model is honest.

The landslide points used in step 8 are **made-up practice data** (see `data/README.md`): they were placed at random, more often on steep slopes. They are not observations.

**Counting lines:** as in Part B, lines that only set up a plot (`fig, ax = plt.subplots()` and `plt.show()`) do not count toward the Q1/Q2/Q3 line counts.

**About the numbers:** reprojection depends on the version of GDAL inside rasterio. With another version, slopes and correlations can differ from the text in the third decimal. This lesson builds each result on the one before. If something looks strange, use **Kernel → Restart Kernel and Run All Cells**: an old variable from an earlier attempt may still be in memory.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 32 of 36 · [Course home](../README.md)  
← Previous: [C8 · Interpolation: a rainfall map from 24 gauges](C08_Interpolation.ipynb) · Next: [D1 · Satellite images and spectral bands: the Melamchi valley from space](../D_Imagery/D01_Satellite_images_and_spectral_bands.ipynb) →

## 1. The idea: are neighbours alike?

The geographer Waldo Tobler put it in one sentence, the **first law of geography** (C8): near things are more alike than distant things. **Spatial autocorrelation** measures how true that is for one variable on one map. *Auto* means *self*: the variable is compared with itself, at nearby places.

- **Positive** autocorrelation: neighbours are alike, high next to high and low next to low. Elevation, rainfall and slope behave like this.
- **No** autocorrelation: a value tells you nothing about its neighbours, as with random noise.
- **Negative** autocorrelation: neighbours tend to differ, like the squares of a chessboard. It is rare in nature.

To measure it, pair every cell with its neighbour to the east, and ask how well one predicts the other. The **correlation coefficient** answers with one number from −1 to +1. Near +1 the pairs lie along a rising line, near 0 they form a shapeless cloud, near −1 they lie along a falling line. `np.corrcoef(a, b)[0, 1]` computes it for two arrays of equal length.

The three 6 × 6 grids below are made-up, one of each kind. `np.indices((6, 6))` returns two grids: the row number and the column number of every cell. `np.random.default_rng(42)` makes a **random number generator** with a fixed **seed**, 42, so its "random" numbers come out the same on every run.

**Predict:** for each grid, will the correlation between a cell and its east neighbour be near +1, near 0 or near −1?

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray as rxr
from rasterio.enums import Resampling

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)

rows, cols = np.indices((6, 6))            # the row number and column number of each cell
checker = (rows + cols) % 2                # 0, 1, 0, 1 ...: a chessboard
rng = np.random.default_rng(42)            # random numbers, the same on every run
noise = rng.integers(0, 11, size=(6, 6))   # whole numbers 0 to 10, scattered at random
smooth = rows + cols                       # rises steadily towards the bottom right
print(checker)
print(noise)
print(smooth)

In [ ]:
def east_pairs(grid):
    """Every cell except the last column, and the cell to its east, as two flat arrays."""
    return grid[:, :-1].ravel(), grid[:, 1:].ravel()

toys = {"Negative: chessboard": checker, "None: random": noise, "Positive: smooth": smooth}
for title, grid in toys.items():
    here, east = east_pairs(grid)
    print(f"{title:21} {len(here)} pairs, correlation {np.corrcoef(here, east)[0, 1]:.2f}")

In [ ]:
# fig is the whole figure; axes holds its 2 x 3 panels, so axes[0, 1] is row 0, column 1
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for col, (title, grid) in enumerate(toys.items()):
    axes[0, col].imshow(grid, cmap="viridis")
    axes[0, col].set_title(title)
    axes[0, col].set_xticks([])
    axes[0, col].set_yticks([])
    here, east = east_pairs(grid)
    axes[1, col].scatter(here, east, alpha=0.4)
    axes[1, col].set_xlabel("Value of a cell")
    axes[1, col].set_ylabel("Value of its east neighbour")
plt.show()

**Predict, answered:** the chessboard gives a correlation of exactly −1.00, the smooth grid exactly +1.00, and the random grid −0.08, close to 0. Each grid has 6 rows of 5 pairs: 30 pairs.

`grid[:, :-1]` drops the last column and `grid[:, 1:]` drops the first, so the two slices line up cell against east neighbour (A8). `.ravel()` lays each slice out as one long 1-D array (C2).

The bottom row of the figure is the idea of this whole lesson. On the chessboard, every 0 has a 1 to its east and every 1 a 0: the dots sit at two corners, on a falling line. On the smooth grid, each east neighbour is exactly one more: the dots form a rising line. Random noise gives a cloud. Its correlation is not exactly 0, because 30 pairs are few and chance alone makes a small correlation; step 6 deals with chance.

### Practice

**Q1 · one line:** Pair each cell of `smooth` with its neighbour to the **south** instead: rows, not columns. Use `smooth[:-1, :]` and `smooth[1:, :]` with `.ravel()`, and print the correlation.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(np.corrcoef(smooth[:-1, :].ravel(), smooth[1:, :].ravel())[0, 1])
```

1.0: going south, each value also grows by exactly one. `[:-1, :]` drops the last row and `[1:, :]` the first, so each cell meets the cell below it.
</details>

**Q2 · two lines:** Make another random 6 × 6 grid with a different seed, `np.random.default_rng(7)`, and print its east-neighbour correlation with `east_pairs`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
here, east = east_pairs(np.random.default_rng(7).integers(0, 11, size=(6, 6)))
print(round(np.corrcoef(here, east)[0, 1], 2))
```

About −0.10: again close to 0, slightly negative. Random grids give small correlations of either sign; only the seed decides which.
</details>

**Q3 · three lines:** Make a grid of vertical stripes with `stripes = cols % 2`. Print its correlation with the east neighbour and with the south neighbour. **Predict** both first.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
stripes = cols % 2
print(np.corrcoef(*east_pairs(stripes))[0, 1])
print(np.corrcoef(stripes[:-1, :].ravel(), stripes[1:, :].ravel())[0, 1])
```

−1.0 to the east, where the stripes flip at every step, and +1.0 to the south, where each stripe repeats. Autocorrelation can depend on **direction**: valleys and ridges that run one way are an example. `*east_pairs(stripes)` unpacks the two returned arrays into the two arguments of `np.corrcoef`.
</details>

## 2. A real raster: correlation with a shifted copy

On a real raster, do the same with every cell, but not only with the next cell: pair each cell with the cell **k** columns to its east, for k = 1, 2, 3 and so on. A shift of k cells compares places k × cell size apart. Plot the correlation against that distance and you get a **correlogram**: how fast the likeness fades with distance.

The grid is the DEM in UTM 45N with **90 m** cells, averaged from the original cells as in C4, so the lesson stays fast. Slope comes from it with `np.gradient` and the true cell size, 90 m (C6). For contrast, `rng.normal` fills a grid of the same shape with random noise.

`grid[:, :-k]` drops the last k columns and `grid[:, k:]` the first k, so the two slices line up k columns apart. The empty edges of the UTM grid hold `NaN` (C4), so the function keeps only the pairs where both values are finite.

The correlogram is the mirror image of C8's variogram. As distance grows, the variogram rises and the correlation falls.

**Predict:** which keeps its likeness over a longer distance, elevation or slope?

In [ ]:
dem = rxr.open_rasterio(DATA / "dem_kathmandu_melamchi.tif", masked=True).squeeze("band", drop=True)
dem_90 = dem.rio.reproject("EPSG:32645", resolution=90, resampling=Resampling.average)   # C4
elev_m = dem_90.values
cell_m = 90
dz_drow, dz_dcol = np.gradient(elev_m, cell_m, cell_m)                                     # C6
slope_deg = np.degrees(np.arctan(np.sqrt(dz_drow ** 2 + dz_dcol ** 2)))
noise_90 = np.random.default_rng(0).normal(size=elev_m.shape)    # random noise on the same grid
print(elev_m.shape, round(float(np.nanmean(slope_deg)), 1), "degrees mean slope")


def shift_corr(grid, k):
    """Correlation between every cell and the cell k columns to its east."""
    here = grid[:, :-k].ravel()
    east = grid[:, k:].ravel()
    both = np.isfinite(here) & np.isfinite(east)        # keep pairs without NaN
    return np.corrcoef(here[both], east[both])[0, 1]


for k in [1, 10, 50]:
    print(f"{k * cell_m:5} m apart: elevation {shift_corr(elev_m, k):.3f}, "
          f"slope {shift_corr(slope_deg, k):.3f}, noise {shift_corr(noise_90, k):.3f}")

In [ ]:
shifts = np.arange(1, 111)                    # 1 to 110 cells: 90 m to 9.9 km
dist_km = shifts * cell_m / 1000
curves = {"Elevation": elev_m, "Slope": slope_deg, "Random noise": noise_90}
correlogram = {}
for name, grid in curves.items():
    correlogram[name] = np.array([shift_corr(grid, k) for k in shifts])

for name in ["Elevation", "Slope"]:
    first_k = shifts[correlogram[name] < 0.5][0]  # the first shift with a correlation below 0.5
    print(f"{name}: correlation drops below 0.5 at {first_k * cell_m:,} m")

fig, ax = plt.subplots(figsize=(8, 4.5))
for name, corr in correlogram.items():
    ax.plot(dist_km, corr, label=name)
ax.axhline(0.5, color="grey", linestyle="--", linewidth=0.8)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Distance between the two cells, west to east (km)")
ax.set_ylabel("Correlation")
ax.set_title("Correlograms on the 90 m DEM grid")
ax.legend()
plt.show()

The 90 m grid has 563 rows and 664 columns, and its mean slope is about 20.8°, gentler than the 23.4° of the 30 m grid in C7: larger cells smooth the terrain (C6).

Next-door cells, 90 m apart, are almost copies: a correlation of about 0.998 for elevation and 0.852 for slope. Elevation stays alike for a long way: the correlation is still about 0.926 at 900 m and falls below 0.5 only at about 5,310 m. Slope loses its likeness much faster, dropping to about 0.54 at 900 m and below 0.5 at about 1,530 m. Random noise stays at 0 at every distance.

Both curves fall steeply at first and then flatten: slope keeps a correlation of about 0.3 even 10 km apart. That long tail is the large-scale pattern of this area: the flat Kathmandu Valley in the south-west and steep mountains in the north and east. Distances like these, 1.5 km for slope and 5 km for elevation, are the scale to think about when you split data for testing in step 8 and D4.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA). The full licence notice is in `data/README.md`.*

### Practice

**Q1 · one line:** What is the correlation of elevation between cells 1.8 km apart (a shift of 20 cells)?

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
shift_corr(elev_m, 20)
```

About 0.822: places 1.8 km apart still share most of their elevation, because they usually lie on the same mountain side or the same valley floor.
</details>

**Q2 · two lines:** At what distance does the correlation of **elevation** first drop below 0.8? Use `correlogram["Elevation"]` and `shifts` as in the example.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
first_k = shifts[correlogram["Elevation"] < 0.8][0]
print(first_k * cell_m, "m")
```

At 2,070 m (23 cells). The mask picks the shifts where the correlation is below 0.8, and `[0]` takes the first of them.
</details>

**Q3 · three lines:** Processing can **create** autocorrelation. Smooth the noise with a 5 × 5 moving mean, `ndimage.uniform_filter(noise_90, size=5)` from `scipy` (C6), and print its correlation at shifts of 1, 4 and 5 cells.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
from scipy import ndimage
blurred = ndimage.uniform_filter(noise_90, size=5)
print([round(shift_corr(blurred, k), 3) for k in [1, 4, 5]])
```

About 0.8, 0.2 and 0.0. Each smoothed cell is the mean of 25 noise cells, and its neighbour shares 20 of them, so the two are alike. Five cells apart, the windows no longer overlap and the likeness is gone. Resampling, smoothing and interpolation (C8) all add autocorrelation of their own.
</details>

## 3. Neighbours for polygons: hexagons and a weights matrix

Raster cells have obvious neighbours: the cells next door. Polygons, such as districts or hexagons, need a rule. The usual rule is **contiguity**: two polygons are neighbours when they share a border. **Queen** contiguity counts any shared point, even a single corner, like the queen's moves in chess. **Rook** contiguity needs a shared edge. For hexagons the two rules agree, because hexagons never meet at a corner only.

Here the polygons are hexagons with **1.5 km** sides, about 5.8 km² each, built with B10's functions over the DEM's area in UTM 45N. Only hexagons wholly inside the DEM are kept, so each holds a full set of cells. `reset_index(drop=True)` numbers them 0, 1, 2 … again, with no gaps.

`gpd.sjoin(hexes, hexes, predicate="touches")` joins the layer to itself (B7, B9). It returns one row per pair of neighbours: the row label of the first hexagon, and the row number of its neighbour in the column `index_right`. This **neighbour table** fills a **weights matrix** W: a NumPy array with one row and one column per hexagon, holding 1 where hexagons i and j are neighbours and 0 everywhere else.

One change from B10: the corners are rounded to the millimetre. Two hexagons compute their shared corner from different centres, and floating-point arithmetic can make the two answers differ by a billionth of a metre. Then `touches` says `False` (B7 warned about this).

In [ ]:
import math
from shapely.geometry import Polygon, box


def hexagon(cx, cy, r):
    """A regular hexagon with centre (cx, cy) and side r, in metres; corners rounded to 1 mm."""
    corners = []
    for angle in range(0, 360, 60):
        corners.append((round(cx + r * math.cos(math.radians(angle)), 3),
                        round(cy + r * math.sin(math.radians(angle)), 3)))
    return Polygon(corners)


def hex_grid(bounds, r):
    """A list of hexagons covering bounds = [xmin, ymin, xmax, ymax] (as in B10)."""
    xmin, ymin, xmax, ymax = bounds
    dx = 1.5 * r
    dy = math.sqrt(3) * r
    hexes = []
    for col, x in enumerate(np.arange(xmin, xmax + dx, dx)):
        shift = dy / 2 if col % 2 == 1 else 0      # every second column moves up
        for y in np.arange(ymin - dy, ymax + dy, dy):
            hexes.append(hexagon(x, y + shift, r))
    return hexes


# The DEM's box in degrees, converted to UTM 45N: the area that has data
dem_area = gpd.GeoSeries([box(85.2, 27.55, 85.8, 28.0)], crs=4326).to_crs(32645).iloc[0]
side_m = 1500
all_hexes = gpd.GeoDataFrame(geometry=hex_grid(dem_area.bounds, side_m), crs=32645)
hexes = all_hexes[all_hexes.within(dem_area)].reset_index(drop=True)
print(len(all_hexes), "hexagons built,", len(hexes), "wholly inside the DEM")
print(round(hexes.area.iloc[0] / 1000000, 2), "km² each")

In [ ]:
pairs = gpd.sjoin(hexes, hexes, predicate="touches")      # the layer joined to itself
print(len(pairs), "rows in the neighbour table")
print(pairs[["index_right"]].head(4))

n = len(hexes)
W = np.zeros((n, n))                         # an n x n grid of zeros
W[pairs.index, pairs["index_right"]] = 1     # a 1 for every pair, as with rows and cols in C7
hexes["n_neighbours"] = W.sum(axis=1).astype(int)    # row sums: neighbours of each hexagon
print(W.shape, int(W.sum()))
print(hexes["n_neighbours"].value_counts().sort_index())

In [ ]:
W_row = W / W.sum(axis=1, keepdims=True)     # divide every row by its own sum
print(W_row[0][W_row[0] > 0])                # the non-zero weights in row 0
print(W_row.sum(axis=1)[:5])                 # every row now adds up to 1

fig, ax = plt.subplots(figsize=(8, 6))
hexes.plot(ax=ax, column="n_neighbours", categorical=True, cmap="Blues", edgecolor="grey", linewidth=0.3,
           legend=True, legend_kwds={"title": "Neighbours", "loc": "upper left", "bbox_to_anchor": (1, 1)})
gpd.GeoSeries([dem_area], crs=32645).boundary.plot(ax=ax, color="black", linewidth=0.8)
ax.set_title("450 hexagons of 1.5 km side and their number of neighbours")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.show()

Of 616 hexagons built over the box, 450 lie wholly inside the DEM, each about 5.85 km². The neighbour table has 2,530 rows, and W holds the same number of 1s. Every pair appears **twice**, once as i → j and once as j → i: if A is B's neighbour, B is A's.

368 hexagons have the full six neighbours. The hexagons along the edge have fewer, from 2 to 5, because part of their ring lies outside the study area. The map shows them as a pale frame.

**Row-standardising** divides every row of W by its own sum, so each row adds up to 1. Hexagon 0 has 3 neighbours, so each gets a weight of 1/3 ≈ 0.333; a hexagon with 6 neighbours gives each 1/6. `W.sum(axis=1)` gives one sum per row (A8); `keepdims=True` keeps those sums as a column of shape (450, 1), so that NumPy divides each row by its **own** sum. Step 4 shows why this is useful.

### Practice

**Q1 · one line:** Which hexagons are the neighbours of hexagon 100? Find the columns of row 100 of `W` that hold a 1, with `np.where` (A8).

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
np.where(W[100] == 1)[0]
```

Six hexagons: 81, 82, 99, 101, 117 and 118. The hexagons are numbered column by column, from the south-west, so 99 and 101 lie directly below and above it in its own column, and the other four in the columns to its west and east.
</details>

**Q2 · two lines:** W is **symmetric**: row i equals column i. `W.T` flips a matrix, so that rows become columns. Check that `W` equals `W.T` everywhere, then do the same for `W_row`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print((W == W.T).all())
print((W_row == W_row.T).all())
```

`True` for W and `False` for W_row. A hexagon on the edge with 3 neighbours gives each of them 1/3, but an inside neighbour with 6 neighbours gives it back only 1/6. Row-standardised weights are no longer symmetric; that is expected.
</details>

**Q3 · three lines:** Squares are different. Make a 3 × 3 block of 1 m squares with `squares = gpd.GeoDataFrame(geometry=[box(x, y, x + 1, y + 1) for x in range(3) for y in range(3)])`, join it to itself with `touches`, and count the neighbours of the centre square (row 4).

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
squares = gpd.GeoDataFrame(geometry=[box(x, y, x + 1, y + 1) for x in range(3) for y in range(3)])
square_pairs = gpd.sjoin(squares, squares, predicate="touches")
print((square_pairs.index == 4).sum())
```

8: the 4 squares that share an edge, and the 4 that share only a corner. `touches` gives **queen** neighbours. Rook neighbours of a square would be only the 4 that share an edge.
</details>

## 4. The spatial lag: the average of the neighbours

The **spatial lag** of a polygon is the average value of its neighbours. With the row-standardised matrix it takes one line: `lag = W_row @ values`.

`@` is **matrix multiplication**, which you met with the affine transform in C1. For each row i, it multiplies every weight in that row by the value of the matching hexagon, and adds up the results. In row i the weights are 1/6 for six neighbours and 0 for all other hexagons, so the sum is the **mean of the six neighbours**. That is why rows that add up to 1 are useful.

The values here are the **mean slope** of each hexagon, from `zonal_stats` (C7) on the 90 m slope grid. If neighbours are alike, the map of the lag looks like a smoother copy of the map of the values.

**Predict:** will the lag map show a wider or a narrower range of slopes than the map of the hexagons themselves?

In [ ]:
from rasterstats import zonal_stats

stats = zonal_stats(hexes, slope_deg, affine=dem_90.rio.transform(), nodata=np.nan, stats=["mean", "count"])
hexes["slope_deg"] = [s["mean"] for s in stats]
hexes["cells"] = [s["count"] for s in stats]
print(hexes["cells"].min(), "to", hexes["cells"].max(), "cells of 90 m per hexagon")
print(hexes["slope_deg"].describe().round(1))

In [ ]:
slope_values = hexes["slope_deg"].to_numpy()
hexes["slope_lag"] = W_row @ slope_values          # mean slope of each hexagon's neighbours

# Check hexagon 0 by hand: the plain mean of its neighbours' values
nb_0 = np.where(W[0] == 1)[0]
print(nb_0, slope_values[nb_0].round(1))
print(round(slope_values[nb_0].mean(), 2), round(hexes.loc[0, "slope_lag"], 2))
print(hexes[["slope_deg", "slope_lag"]].describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))       # one figure, two maps side by side
titles = ["Mean slope of each hexagon", "Spatial lag: mean slope of its neighbours"]
for ax, column, title in zip(axes, ["slope_deg", "slope_lag"], titles):
    hexes.plot(ax=ax, column=column, cmap="YlOrRd", vmin=0, vmax=40, edgecolor="white", linewidth=0.2,
               legend=True, legend_kwds={"label": "Slope (degrees)", "shrink": 0.7})
    ax.set_title(title)
    ax.set_axis_off()
plt.show()

Each hexagon holds between 707 and 725 cells of 90 m. Mean slopes range from about 1.6° on the floor of the Kathmandu Valley to about 37.2° in the mountains, with an average of about 20.2°.

Hexagon 0, in the south-west corner, has three neighbours (hexagons 1, 18 and 19), with mean slopes of about 16.9°, 30.2° and 19.0°. Their plain mean, about 22.04°, is exactly its spatial lag: `W_row @` is "the mean of the neighbours" for all 450 hexagons at once.

**Predict, answered:** the lag ranges only from about 2.2° to 34.1°, while the hexagons themselves range from 1.6° to 37.2°, and its standard deviation is smaller (about 6.8° against 7.7°). Averaging always pulls extremes towards the middle. Yet the two maps look almost the same, with the gentle valley in the south-west and steep ground in the north and east: hexagons resemble their neighbours.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA). The full licence notice is in `data/README.md`.*

### Practice

**Q1 · one line:** How closely does a hexagon follow its neighbours? Print the correlation (step 1) between `hexes["slope_deg"]` and `hexes["slope_lag"]`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
np.corrcoef(hexes["slope_deg"], hexes["slope_lag"])[0, 1]
```

About 0.924: knowing the neighbours' mean slope tells you most of a hexagon's own slope. Remember this number for step 8.
</details>

**Q2 · two lines:** Compute the mean **elevation** of each hexagon with `zonal_stats` on `elev_m`, store it in `hexes["elev_m"]`, and store its spatial lag in `hexes["elev_lag"]`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
hexes["elev_m"] = [s["mean"] for s in zonal_stats(hexes, elev_m, affine=dem_90.rio.transform(), nodata=np.nan, stats=["mean"])]
hexes["elev_lag"] = W_row @ hexes["elev_m"].to_numpy()
```

The same two steps as for slope: zonal means, then `W_row @`. Run `hexes[["elev_m", "elev_lag"]].describe()` to compare: the hexagons' mean elevations run from about 613 m to 3,384 m, the lag only from about 722 m to 3,232 m. Averaging over neighbours narrows the range again.
</details>

**Q3 · three lines:** Which hexagon differs most from its neighbours? Subtract the lag from the slope, find the row with the largest absolute difference with `.abs().idxmax()` (A9), and print that row's `slope_deg`, `slope_lag` and `n_neighbours`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
diff_deg = hexes["slope_deg"] - hexes["slope_lag"]
i = diff_deg.abs().idxmax()
print(i, hexes.loc[i, ["slope_deg", "slope_lag", "n_neighbours"]])
```

Hexagon 43, at about 27.5° among six neighbours averaging only about 16.8°: a steep patch, such as a ridge or a gorge side, among gentler ground. `idxmax` returns the row label of the largest value. Such outliers return in step 7.
</details>

## 5. Moran's I: one number for the whole map

**Moran's I** sums up spatial autocorrelation in one number. It works with **deviations** from the mean, z = value − mean: above-average hexagons get a positive z and below-average ones a negative z.

I = (n / S₀) × Σᵢ Σⱼ wᵢⱼ zᵢ zⱼ / Σᵢ zᵢ²

- Σⱼ wᵢⱼ zⱼ is the spatial lag of the deviations: the neighbours' average deviation. The product zᵢ × lag is **positive** when a hexagon and its neighbours lie on the same side of the mean, and **negative** when they lie on opposite sides. Moran's I adds up these products.
- Dividing by Σᵢ zᵢ² scales the result, as a correlation is scaled. n is the number of hexagons and S₀ the sum of all weights. With row-standardised weights S₀ = n, so n / S₀ = 1.

I near +1 means strong positive autocorrelation, near 0 none, and negative values mean that neighbours differ. With no pattern at all, the expected value is −1 / (n − 1): almost zero.

The **Moran scatter plot** draws each hexagon's **standardised** value (its deviation divided by the standard deviation) against the spatial lag of those values. It is step 1's neighbour scatter plot, for polygons. With row-standardised weights, the slope of the best straight line through it equals Moran's I exactly.

In [ ]:
def morans_i(values, w):
    """Moran's I of a 1-D array of values, with a weights matrix w."""
    z = values - values.mean()         # deviations from the mean
    lag_z = w @ z                       # the neighbours' (weighted) deviations
    n = len(values)
    return (n / w.sum()) * (z * lag_z).sum() / (z ** 2).sum()


moran_slope = morans_i(slope_values, W_row)
print(round(moran_slope, 3), "Moran's I of mean slope")
print(round(-1 / (n - 1), 4), "expected with no pattern")

In [ ]:
z_std = (slope_values - slope_values.mean()) / slope_values.std()   # standardised values
lag_std = W_row @ z_std
a, b = np.polyfit(z_std, lag_std, 1)       # best straight line (C8)
print(round(a, 3), "slope of the line")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(z_std, lag_std, s=12, alpha=0.5)
line = np.array([z_std.min(), z_std.max()])
ax.plot(line, a * line + b, color="crimson", label=f"Fitted line, slope {a:.3f}")
ax.axhline(0, color="grey", linewidth=0.8)
ax.axvline(0, color="grey", linewidth=0.8)
ax.set_xlabel("Mean slope of the hexagon (standardised)")
ax.set_ylabel("Mean of its neighbours (standardised)")
ax.set_title("Moran scatter plot, 450 hexagons")
ax.legend()
plt.show()

Moran's I of the mean slope is about **0.817**, against about −0.0022 expected without a pattern. That is strong positive autocorrelation: steep hexagons have steep neighbours, gentle hexagons have gentle neighbours.

The fitted line has a slope of 0.817 too, the same number: Moran's I **is** the slope of the Moran scatter plot. Most dots lie in the lower-left and upper-right corners, where a hexagon and its neighbours sit on the same side of the mean. The few dots in the other two corners are hexagons unlike their neighbours.

`morans_i` works with any weights matrix: it does not assume that rows add up to 1, because it divides by `w.sum()`. In real projects, the **PySAL** family of libraries (`libpysal` for weights, `esda` for Moran's I) does all of this in a few lines and offers more options. You have now built it by hand, so you know what they compute.

### Practice

**Q1 · one line:** Compute Moran's I of the mean slope with the plain 0/1 matrix `W` instead of `W_row`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
morans_i(slope_values, W)
```

About 0.822, very close to 0.817. The `n / w.sum()` factor rescales the result, so the two kinds of weights give similar answers; they differ only in how much the edge hexagons, with fewer neighbours, count.
</details>

**Q2 · two lines:** Give every hexagon a random value, `noise_hex = np.random.default_rng(1).normal(size=n)`, and compute its Moran's I.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
noise_hex = np.random.default_rng(1).normal(size=n)
print(round(morans_i(noise_hex, W_row), 3))
```

About −0.028: close to the expected −0.002, as it should be for values with no pattern. It is not exactly zero; step 6 asks whether a value like 0.817 could arise the same way.
</details>

**Q3 · three lines:** Compute Moran's I of the mean **elevation** per hexagon. Get the means from `zonal_stats` on `elev_m` (as in step 4), turn them into an array, and call `morans_i`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
elev_stats = zonal_stats(hexes, elev_m, affine=dem_90.rio.transform(), nodata=np.nan, stats=["mean"])
elev_values = np.array([s["mean"] for s in elev_stats])
print(round(morans_i(elev_values, W_row), 3))
```

About 0.812: elevation per hexagon is about as strongly autocorrelated as slope. At the 30 m scale of step 2, elevation stayed alike much longer than slope did; averaged over 5.8 km² hexagons, both are dominated by the same large pattern of valley and mountains.
</details>

## 6. Could it be chance? A permutation test

Random values give a Moran's I a little above or below zero (Q2 of step 5). Is 0.817 real, or could chance produce it? A **permutation test** answers by experiment.

Keep the hexagons and their neighbours, but **shuffle** the slope values among them at random. A shuffle destroys any pattern in space but keeps the same values. Compute I for the shuffled map, and repeat 999 times. The 999 results show what Moran's I looks like when there is **no** spatial pattern.

The **pseudo p-value** is (number of shuffles with I at least as large as observed + 1) / (999 + 1). The + 1 counts the observed map itself as one possible arrangement. A small p-value, such as 0.001, means that not one shuffle in 999 came close: chance is a poor explanation. 0.001 is also the smallest value that 999 shuffles can give.

`rng.permutation(values)` returns a shuffled copy. In a loop that does not need its loop variable, Python programmers write `_` as its name. The seeded `rng` is made in the same cell, so the 999 shuffles are the same on every run.

In [ ]:
rng = np.random.default_rng(42)
shuffled_slope = rng.permutation(slope_values)       # same values, random places
print(round(morans_i(shuffled_slope, W_row), 3), "Moran's I after one shuffle")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, values, title in zip(axes, [slope_values, shuffled_slope], ["Real map", "The same values, shuffled"]):
    hexes.plot(ax=ax, column=values, cmap="YlOrRd", vmin=0, vmax=40, edgecolor="white", linewidth=0.2,
               legend=True, legend_kwds={"label": "Slope (degrees)", "shrink": 0.7})
    ax.set_title(title)
    ax.set_axis_off()
plt.show()

In [ ]:
rng = np.random.default_rng(42)
shuffled_i = np.array([morans_i(rng.permutation(slope_values), W_row) for _ in range(999)])
at_least = (shuffled_i >= moran_slope).sum()
p_value = (at_least + 1) / (999 + 1)
print("shuffled I: mean", round(shuffled_i.mean(), 4), "min", round(shuffled_i.min(), 3), "max", round(shuffled_i.max(), 3))
print(at_least, "shuffles reached", round(moran_slope, 3), "; pseudo p-value", p_value)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(shuffled_i, bins=30, color="lightgrey", edgecolor="grey", label="999 shuffled maps")
ax.axvline(moran_slope, color="crimson", linewidth=2, label=f"Real map, I = {moran_slope:.3f}")
ax.set_xlabel("Moran's I")
ax.set_ylabel("Number of shuffled maps")
ax.legend()
plt.show()

One shuffle gives an I of about −0.039, and its map looks like confetti: the same 450 values, but no valley and no mountains. `hexes.plot(column=values)` accepts an array instead of a column name, as long as it has one value per row.

The 999 shuffles give Moran's I values from about −0.10 to 0.11, centred on about −0.002, the expected value. Not one reaches 0.817, so the pseudo p-value is 0.001, the smallest possible. The pattern in the slope map is not chance.

Be clear about what the test says. It says the hexagons are **not arranged at random**. It does not say why: here the reason is the geology and the rivers that shaped the valley and the mountains.

### Practice

**Q1 · one line:** Between which two values do the middle 95% of the shuffled I values lie? Use `np.percentile` (C2) with `[2.5, 97.5]`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
np.percentile(shuffled_i, [2.5, 97.5])
```

From about −0.06 to 0.06. Any Moran's I outside this band would be unusual for a map without pattern; 0.817 is far outside it.
</details>

**Q2 · two lines:** Repeat the test with only **99** shuffles, and print the pseudo p-value. What is the smallest value 99 shuffles can give?

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
few_i = np.array([morans_i(rng.permutation(slope_values), W_row) for _ in range(99)])
print(((few_i >= moran_slope).sum() + 1) / (99 + 1))
```

0.01, the smallest possible with 99 shuffles: (0 + 1) / 100. The number of shuffles sets a floor under the p-value, which is why 999 is a common choice.
</details>

**Q3 · three lines:** Test the random hexagon values from step 5: make `noise_hex` again with `np.random.default_rng(1).normal(size=n)`, compute 999 shuffled I values, and print the observed I and its pseudo p-value.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
noise_hex = np.random.default_rng(1).normal(size=n)
noise_i = np.array([morans_i(rng.permutation(noise_hex), W_row) for _ in range(999)])
print(round(morans_i(noise_hex, W_row), 3), ((noise_i >= morans_i(noise_hex, W_row)).sum() + 1) / 1000)
```

The observed I is about −0.028 and the p-value is large, about 0.8: most shuffles give a larger I. Random values pass as random. The exact p-value depends on how often `rng` has been used before, so yours may differ a little.
</details>

## 7. The local view: clusters and outliers

Moran's I is **global**: one number for the whole map. It cannot say *where* the clusters are. The Moran scatter plot can, because each dot is one hexagon. Its two zero lines split it into four **quadrants**:

- **high-high**: above average, among above-average neighbours: a cluster of steep ground,
- **low-low**: below average, among below-average neighbours: a cluster of gentle ground,
- **high-low** and **low-high**: a hexagon unlike its neighbours, a spatial **outlier**.

Mapping the quadrant of every hexagon shows where the clusters lie. This is a **simplified** version of **local Moran's I**, also called **LISA** (local indicators of spatial association). The full method also runs a permutation test for every hexagon and greys out those that could be chance. The map below has **no** such test: read it as a description, not as proof.

`np.select` (C5) gives each hexagon a label from a list of conditions.

In [ ]:
conditions = [(z_std > 0) & (lag_std > 0), (z_std < 0) & (lag_std < 0),
              (z_std > 0) & (lag_std < 0), (z_std < 0) & (lag_std > 0)]
labels = ["high-high", "low-low", "high-low", "low-high"]
hexes["quadrant"] = np.select(conditions, labels, default="none")
print(hexes["quadrant"].value_counts())

In [ ]:
from matplotlib.colors import ListedColormap

districts = gpd.read_file(DATA / "nepal_districts.geojson").to_crs(32645)
near = gpd.clip(districts, dem_area)                  # district borders inside the DEM area (B7)

# geopandas sorts the labels: high-high, high-low, low-high, low-low; one colour each
quadrant_colours = ListedColormap(["#d7191c", "#fdae61", "#abd9e9", "#2c7bb6"])
fig, ax = plt.subplots(figsize=(9, 7))
hexes.plot(ax=ax, column="quadrant", categorical=True, cmap=quadrant_colours, edgecolor="white",
           linewidth=0.2, legend=True, legend_kwds={"title": "Quadrant", "loc": "upper left", "bbox_to_anchor": (1, 1)})
near.boundary.plot(ax=ax, color="black", linewidth=0.6)
for name, point in zip(near["district"], near.representative_point()):
    if name in ["Kathmandu", "Lalitpur", "Bhaktapur", "Sindhupalchok", "Nuwakot", "Kabhrepalanchok"]:
        ax.annotate(name, (point.x, point.y), ha="center", fontsize=9)
ax.set_title("Simplified local Moran map of mean slope (no significance test)")
ax.set_axis_off()
plt.show()

222 hexagons are high-high and 157 low-low: 379 of 450, about 84%, sit among neighbours like themselves. Only 36 are high-low and 35 low-high.

The map shows one large **low-low** cluster, from the Kathmandu Valley east into Kabhrepalanchok, and a smaller one in Nuwakot in the north-west. The **high-high** ground covers the hills and mountains to the north and east. The outliers sit mostly along the borders between the two, where a gentle hexagon touches steep ones, or the reverse.

`representative_point()` (B11) puts each name inside its district; `gpd.clip` cuts the district borders to the DEM area, so the map stays on the study area. The colours follow the usual convention for such maps: red for high-high, blue for low-low, paler shades for the outliers.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA). The full licence notice is in `data/README.md`.*

### Practice

**Q1 · one line:** What share of the hexagons lies in a cluster, high-high or low-low? Use `.isin` (B5) and `.mean()`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
hexes["quadrant"].isin(["high-high", "low-low"]).mean()
```

About 0.842, or 84%. The mean of a True/False column is the share of `True` values.
</details>

**Q2 · two lines:** Print the number of hexagons and the mean slope in each quadrant with `groupby` and `.agg(["count", "mean"])` (A9).

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
quadrant_table = hexes.groupby("quadrant")["slope_deg"].agg(["count", "mean"])
print(quadrant_table.round(1))
```

High-high hexagons average about 26.0° and low-low about 11.8°; the outliers lie in between, high-low at about 22.2° and low-high at about 18.8°. An outlier is only high or low **relative to the mean**, 20.2°, not steep or gentle in absolute terms.
</details>

**Q3 · three lines:** In which districts do the **low-low** hexagons lie? Make points at the hexagon centres with `hexes.centroid`, keep the low-low ones, join them to `districts` with `sjoin` (B9, `predicate="within"`) and count them per district.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
centres = gpd.GeoDataFrame(hexes[["quadrant"]], geometry=hexes.centroid, crs=32645)
low_low = centres[centres["quadrant"] == "low-low"]
print(gpd.sjoin(low_low, districts, predicate="within")["district"].value_counts())
```

Kathmandu (51) and Kabhrepalanchok (50) hold the most, then Bhaktapur (21) and Lalitpur (19), with a few in Nuwakot (9) and Sindhupalchok (7). The gentle cluster runs from the Kathmandu Valley east into Kabhrepalanchok. `.centroid` turns each hexagon into a point, so each is counted in exactly one district.
</details>

## 8. Why it matters: landslides and honest tests

The practice landslides in `practice_landslides_melamchi.csv` are **made-up**: they were placed at random, more often on slopes steeper than about 15° (C7). Slope is strongly autocorrelated, so the number of landslides per hexagon should be autocorrelated too, more weakly.

This matters as soon as you test a model. A model learns from **training** data and is scored on **test** data it has not seen (C8). If you split hexagons into training and test **at random**, almost every test hexagon has a training neighbour, and neighbours are near-copies: step 4 found a correlation of 0.924 between a hexagon and its neighbours' mean. A test like that rewards a model for copying its neighbours. Its score looks better than it would be in a new area, where no neighbour is known.

The fix is a **spatial split**: hold out whole regions, such as the eastern fifth of the map. The second cell compares the two splits with a deliberately naive "model": predict the mean slope of each test hexagon by copying the value of the **nearest training hexagon**. D4 does the same comparison with a real machine-learning model.

**Predict:** which split will give the naive model the smaller error?

In [ ]:
slides = pd.read_csv(DATA / "practice_landslides_melamchi.csv")        # made-up practice data
landslides = gpd.GeoDataFrame(slides, geometry=gpd.points_from_xy(slides["lon"], slides["lat"]),
                              crs="EPSG:4326").to_crs(32645)
in_hex = gpd.sjoin(landslides, hexes[["geometry"]], predicate="within")   # B9
hexes["n_slides"] = in_hex.groupby("index_right").size()    # lines up by row label (C7)
hexes["n_slides"] = hexes["n_slides"].fillna(0).astype(int) # hexagons without landslides: 0
print(len(in_hex), "of", len(landslides), "landslides lie in a hexagon")
print(hexes["n_slides"].value_counts().sort_index())

count_values = hexes["n_slides"].to_numpy().astype(float)
moran_slides = morans_i(count_values, W_row)
rng = np.random.default_rng(42)
shuffled = np.array([morans_i(rng.permutation(count_values), W_row) for _ in range(999)])
print("Moran's I of landslide counts:", round(moran_slides, 3),
      "; pseudo p-value", ((shuffled >= moran_slides).sum() + 1) / 1000)
print("correlation of counts with mean slope:", round(np.corrcoef(count_values, slope_values)[0, 1], 3))

In [ ]:
centre_x = hexes.centroid.x.to_numpy()
centre_y = hexes.centroid.y.to_numpy()


def nearest_copy(is_test, values):
    """Copy each test hexagon's value from its nearest training hexagon.
    Returns the RMSE and the mean distance to that training hexagon in km."""
    train = np.where(~is_test)[0]
    errors, gaps_km = [], []
    for i in np.where(is_test)[0]:
        dist_m = np.sqrt((centre_x[train] - centre_x[i]) ** 2 + (centre_y[train] - centre_y[i]) ** 2)
        errors.append(values[train[dist_m.argmin()]] - values[i])
        gaps_km.append(dist_m.min() / 1000)
    return np.sqrt(np.mean(np.array(errors) ** 2)), np.mean(gaps_km)


random_test = np.random.default_rng(42).random(n) < 0.2       # about 20% of hexagons, at random
east_test = centre_x > np.percentile(centre_x, 80)              # the eastern 20%, one block
for name, is_test in [("random split", random_test), ("eastern block", east_test)]:
    rmse_deg, gap_km = nearest_copy(is_test, slope_values)
    baseline_deg = np.sqrt(np.mean((slope_values[~is_test].mean() - slope_values[is_test]) ** 2))
    with_nb = (W[is_test][:, ~is_test].sum(axis=1) > 0).mean() * 100
    print(f"{name}: {is_test.sum()} test hexagons, {with_nb:.0f}% touch a training hexagon, "
          f"nearest training hexagon {gap_km:.1f} km away, RMSE {rmse_deg:.1f}° (baseline {baseline_deg:.1f}°)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, is_test, title in zip(axes, [random_test, east_test], ["Random split", "Spatial split: eastern block"]):
    hexes.plot(ax=ax, color=np.where(is_test, "crimson", "lightgrey"), edgecolor="white", linewidth=0.2)
    landslides.plot(ax=ax, color="black", markersize=6)
    ax.set_title(f"{title}: test hexagons in red")
    ax.set_axis_off()
plt.show()

53 of the 60 made-up landslides fall in a hexagon; the other 7 lie in the edge strip that no whole hexagon covers. 401 hexagons hold none, 45 hold one and 4 hold two. Their Moran's I is small, about 0.049, but the permutation test gives a pseudo p-value of about 0.029: weak, but unlikely to be chance. The counts correlate with mean slope (about 0.224), and slope is strongly autocorrelated, so some of that pattern carries over to the counts.

**Predict, answered:** with the random split, all 103 test hexagons touch a training hexagon, and the nearest one is always the next hexagon, 2.6 km away. Copying it gives an RMSE of about 4.0°, against 7.3° for the baseline that ignores location (the training mean, C8): the naive model looks about 45% better than the baseline. With the eastern block, the nearest training hexagon is on average 6.9 km away, and only 20% of the test hexagons, those along the block's western edge, touch one. The RMSE grows to about 6.6° against a baseline of 8.1°: only about 19% better.

Same model, same data: only the split changed. The random split measured how well the model copies neighbours; the spatial split measures how well it works somewhere new, which is what a hazard map is for. On the map, red test hexagons in the random split are scattered among grey training ones; in the spatial split they form one block.

### Practice

**Q1 · one line:** Print the mean slope of the hexagons **with** landslides and of those **without**, using `count_values` and `slope_values`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(slope_values[count_values > 0].mean(), slope_values[count_values == 0].mean())
```

About 24.9° with landslides and 19.6° without. The made-up points favour steep ground, as C7 showed with the frequency ratio.
</details>

**Q2 · two lines:** Was seed 42 a lucky draw? Make random splits with seeds 0 to 9 in a list comprehension (A3), keep the RMSE of `nearest_copy` on `slope_values` for each, and print the smallest, the largest and the mean.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
rmses = [nearest_copy(np.random.default_rng(s).random(n) < 0.2, slope_values)[0] for s in range(10)]
print(round(min(rmses), 1), round(max(rmses), 1), round(np.mean(rmses), 1))
```

From about 4.0° to 5.8°, 4.7° on average: every random split stays below the eastern block's 6.6°. The optimism comes from random splitting itself, not from one lucky seed. `[0]` takes the first of the two values that `nearest_copy` returns, the RMSE.
</details>

**Q3 · three lines:** Try a block on the other side: the **western** 20% of hexagons (`centre_x < np.percentile(centre_x, 20)`). Print its RMSE and mean distance with `nearest_copy`, and its baseline RMSE.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
west_test = centre_x < np.percentile(centre_x, 20)
rmse_deg, gap_km = nearest_copy(west_test, slope_values)
print(round(rmse_deg, 1), round(gap_km, 1), round(np.sqrt(np.mean((slope_values[~west_test].mean() - slope_values[west_test]) ** 2)), 1))
```

An RMSE of about 9.7° with the nearest training hexagon 6.9 km away on average, against a baseline of about 8.0°: here copying the nearest known place is **worse** than using the plain training mean. At 6.9 km, the correlogram of step 2 says slope is only weakly alike, so the nearest training hexagon often lies on quite different ground. A spatial score also depends on which block you hold out, so D4 holds out many blocks in turn and averages.
</details>

## 9. When it breaks: row labels that are not 0, 1, 2 …

In step 3, `reset_index(drop=True)` renumbered the hexagons after the selection. Here it is left out. The table still looks fine, but run the cell.

In [ ]:
inside = all_hexes[all_hexes.within(dem_area)]          # no reset_index this time
pairs_bad = gpd.sjoin(inside, inside, predicate="touches")
W_bad = np.zeros((len(inside), len(inside)))
W_bad[pairs_bad.index, pairs_bad["index_right"]] = 1

Read the last line: `IndexError: index … is out of bounds for axis 0 with size 450`.

- `inside` keeps the **row labels** of `all_hexes`, which run up to 527 with gaps: the hexagons outside the DEM were dropped, their labels with them.
- `W_bad` has only 450 rows, numbered 0 to 449. A label such as 500 has no row there.

A pandas row **label** and a NumPy **position** are different things (A9: `.loc` against `.iloc`). After filtering, they no longer agree. NumPy needs positions, so renumber the rows first with `reset_index(drop=True)`.

The dangerous case is the one without an error. If every label happened to stay below 450, NumPy would fill the wrong rows silently, and Moran's I would be wrong. Renumber the rows **every time** you filter before building a weights matrix.

**Try it:** add `.reset_index(drop=True)` to the end of the first line and run the cell again.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · Moran's I of the three toy grids

Go back to the 6 × 6 grids of step 1. Build a 36 × 36 **rook** weights matrix: cell (r, c) is number `r * 6 + c`, and its neighbours are the cells above, below, left and right that lie inside the grid. Row-standardise it, then print Moran's I of `checker`, `noise` and `smooth` (flattened with `.ravel()`).

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
W_toy = np.zeros((36, 36))
for r in range(6):
    for c in range(6):
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            if 0 <= r + dr < 6 and 0 <= c + dc < 6:
                W_toy[r * 6 + c, (r + dr) * 6 + (c + dc)] = 1
W_toy_row = W_toy / W_toy.sum(axis=1, keepdims=True)
for title, grid in toys.items():
    print(f"{title:21} Moran's I {morans_i(grid.ravel().astype(float), W_toy_row):.3f}")
```

The chessboard gives exactly −1.000: every neighbour of a 1 is a 0, and the reverse. The random grid gives about −0.044, near the expected −1/35 ≈ −0.029. The smooth grid gives 0.889, not 1: inside the grid, the four neighbours average exactly to the cell's own value, but at the edges they all lie on one side. Cell (0, 0) holds 0, and both its neighbours hold 1. With a few cells, edges matter a lot. `r * 6 + c` is the position of cell (r, c) in the flattened grid, the same order that `.ravel()` uses.
</details>

### Review 2 · Does the hexagon size matter?

Repeat the analysis with hexagons of **3 km** side instead of 1.5 km: build them, keep those wholly inside `dem_area`, renumber, build the neighbour table and the row-standardised matrix, compute the mean slope per hexagon, and print the number of hexagons and Moran's I.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
big = gpd.GeoDataFrame(geometry=hex_grid(dem_area.bounds, 3000), crs=32645)
big = big[big.within(dem_area)].reset_index(drop=True)
big_pairs = gpd.sjoin(big, big, predicate="touches")
W_big = np.zeros((len(big), len(big)))
W_big[big_pairs.index, big_pairs["index_right"]] = 1
W_big_row = W_big / W_big.sum(axis=1, keepdims=True)
big_slope = np.array([s["mean"] for s in zonal_stats(big, slope_deg, affine=dem_90.rio.transform(), nodata=np.nan, stats=["mean"])])
print(len(big), "hexagons; Moran's I", round(morans_i(big_slope, W_big_row), 3))
```

102 hexagons, and Moran's I falls from 0.817 to about 0.757. The centres of neighbouring hexagons are now about 5.2 km apart instead of 2.6 km, and step 2 showed that likeness fades with distance. Moran's I is not a property of the terrain alone: it depends on the size and shape of the units you measure it with (B10 saw the same for counts). Always report them.
</details>

### Review 3 · Neighbours by distance

Contiguity is one rule; **distance** is another. Make a weights matrix in which hexagon j is a neighbour of hexagon i when their centres are more than 0 and at most 5.5 km apart: the first two rings of hexagons. Use `centre_x` and `centre_y` from step 8 and a loop over the rows. Row-standardise it, and print the mean number of neighbours and Moran's I of `slope_values`.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
W_band = np.zeros((n, n))
for i in range(n):
    dist_m = np.sqrt((centre_x - centre_x[i]) ** 2 + (centre_y - centre_y[i]) ** 2)
    W_band[i] = (dist_m > 0) & (dist_m <= 5500)
W_band_row = W_band / W_band.sum(axis=1, keepdims=True)
print(round(W_band.sum(axis=1).mean(), 1), "neighbours on average")
print(round(morans_i(slope_values, W_band_row), 3), "Moran's I with a 5.5 km band")
print(round(morans_i(slope_values, W_row), 3), "Moran's I with touching hexagons")
```

Each hexagon now has about 16.2 neighbours on average (two full rings hold 18), and Moran's I drops to about 0.718. The second ring lies about 4.5 to 5.2 km away, where hexagons are less alike. The weights are a choice, like a threshold or a block size: state the rule you used.
</details>

### Review 4 · Save the hexagons and read them back

Save `hexes` with the columns `slope_deg`, `slope_lag`, `quadrant`, `n_slides` and the geometry to `../outputs/c09_hexagons.gpkg` (B3). Read the file back, and print its number of rows, its CRS and the number of hexagons per quadrant. Check that Moran's I of the `slope_deg` column read back is still 0.817.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
columns = ["slope_deg", "slope_lag", "quadrant", "n_slides", "geometry"]
hexes[columns].to_file(OUT / "c09_hexagons.gpkg", layer="hexagons", driver="GPKG")

back = gpd.read_file(OUT / "c09_hexagons.gpkg", layer="hexagons")
print(len(back), back.crs.to_epsg())
print(back["quadrant"].value_counts())
print(round(morans_i(back["slope_deg"].to_numpy(), W_row), 3))
```

450 hexagons in EPSG:32645, with the same quadrant counts and the same Moran's I, 0.817. The file keeps the hexagons in the order they were written, so `W_row` still matches its rows. A weights matrix is only valid for rows in the same order as when it was built: sort or filter the table, and you must build it again.
</details>

## Recap

- **Spatial autocorrelation** measures how alike nearby values are: positive (alike), none (random) or negative (opposite). Pairs of neighbours and a **correlation coefficient** show it.
- On a raster, correlate the grid with a copy shifted by k cells: the **correlogram** shows how far the likeness reaches. Here slope fell below 0.5 at about 1.5 km and elevation at about 5.3 km.
- For polygons, a **neighbour table** (`sjoin` with `touches`) fills a **weights matrix**; **row-standardised**, each row adds up to 1 and `W_row @ values` is the **spatial lag**, the mean of the neighbours. Renumber rows with `reset_index(drop=True)` first.
- **Moran's I** is the slope of the Moran scatter plot; a **permutation test** with 999 shuffles gives a pseudo p-value. Mean slope per hexagon: I ≈ 0.817, p = 0.001. The four quadrants map clusters and outliers, without a significance test in this simple form.
- Autocorrelated data make **random** train/test splits too optimistic, because test samples have near-copies in training. A **spatial** split keeps whole regions apart; D4 uses it for a real model. Libraries such as PySAL (`libpysal`, `esda`) do all this for real projects.

### Words you learned

| Word | Meaning |
| --- | --- |
| first law of geography | Tobler's rule: near things are more alike than distant things |
| spatial autocorrelation | How much the values of one variable at nearby places resemble each other |
| positive / negative autocorrelation | Neighbours alike / neighbours unlike |
| correlation coefficient | A number from −1 to +1 that says how well two sets of values follow a straight line together |
| random number generator, seed | A source of random numbers; a fixed seed gives the same numbers every run |
| correlogram | Correlation plotted against the distance between the paired places |
| contiguity | Being neighbours by sharing a border |
| queen / rook contiguity | Neighbours that share any point, even a corner / that share an edge |
| neighbour table | A table with one row for each pair of neighbouring polygons |
| weights matrix (W) | An n × n array with a weight for every pair of polygons, 0 for non-neighbours |
| row-standardised | Each row of W divided by its sum, so that every row adds up to 1 |
| symmetric matrix | A matrix equal to its own flip (`W.T`): row i equals column i |
| spatial lag | The (weighted) mean of a polygon's neighbours' values |
| matrix multiplication (`@`) | For every row: multiply each weight by the matching value and add them up |
| deviation | A value minus the mean |
| standardised value | A deviation divided by the standard deviation |
| Moran's I | One number for the spatial autocorrelation of a whole map |
| Moran scatter plot | Standardised values against their spatial lag; its slope is Moran's I |
| permutation test | Shuffling values many times to see what results chance alone gives |
| pseudo p-value | (shuffles at least as extreme as observed + 1) / (shuffles + 1) |
| global / local statistic | One number for the whole map / one number for each place |
| LISA, local Moran's I | Local indicators of spatial association: clusters and outliers, normally tested for significance |
| cluster / spatial outlier | A place like its neighbours / a place unlike its neighbours |
| training / test data | The data a model learns from / the data it is scored on |
| spatial split | A train/test split that keeps whole regions apart |
| PySAL | A family of Python libraries for spatial statistics (`libpysal`, `esda` and others) |

## Next

This is the end of Part C. **Part D · Satellite imagery** begins with **D1 · Satellite images and spectral bands**: rasters with many bands, one for each kind of light. In D4 you will meet this lesson's idea again, when a spatial split tests a land-cover model honestly.

<!-- pygile-nav -->
**PyGILE Course** · lesson 32 of 36 · [Course home](../README.md)  
← Previous: [C8 · Interpolation: a rainfall map from 24 gauges](C08_Interpolation.ipynb) · Next: [D1 · Satellite images and spectral bands: the Melamchi valley from space](../D_Imagery/D01_Satellite_images_and_spectral_bands.ipynb) →